# 📊 Sorting, Searching & Heaps

Sorting is a preprocessing step for many algorithms. Heaps give you efficient
access to the minimum/maximum element. Together they solve a huge class of problems.

**Recognition Triggers:**
- "merge overlapping intervals" → Sort by start
- "K largest / K smallest" → Heap
- "median from stream" → Two heaps
- "merge K sorted lists" → Heap

In [ ]:
import heapq
from collections import Counter, defaultdict

---
## 📋 Sorting Algorithms

In [ ]:
# Merge Sort — O(n log n), stable, good for linked lists
def merge_sort(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

# Quick Sort — O(n log n) average, O(n²) worst, in-place
def quick_sort(arr):
    if len(arr) <= 1: return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quick_sort(left) + middle + quick_sort(right)

print("Merge Sort:", merge_sort([38, 27, 43, 3, 9, 82, 10]))
print("Quick Sort:", quick_sort([38, 27, 43, 3, 9, 82, 10]))

In [ ]:
# Custom sorting in Python
from functools import cmp_to_key

# Sort by key
intervals = [[1,3],[2,6],[8,10],[15,18]]
intervals_sorted = sorted(intervals, key=lambda x: x[0])
print("Sort by start:", intervals_sorted)

# Sort by multiple keys
students = [('Alice', 85), ('Bob', 90), ('Charlie', 85)]
students_sorted = sorted(students, key=lambda x: (-x[1], x[0]))  # By grade desc, then name
print("Multi-key sort:", students_sorted)

# Custom comparator (for Largest Number problem)
def compare(a, b):
    if a + b > b + a: return -1
    elif a + b < b + a: return 1
    return 0

nums = ['3', '30', '34', '5', '9']
nums.sort(key=cmp_to_key(compare))
print("Largest Number:", ''.join(nums))  # "9534330"

---
## 🔥 Problem: Merge Intervals

In [ ]:
def merge_intervals(intervals):
    """Sort by start, merge overlapping. Time: O(n log n)"""
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

print("Merge Intervals:")
print(" ", merge_intervals([[1,3],[2,6],[8,10],[15,18]]))  # [[1,6],[8,10],[15,18]]

In [ ]:
# Insert Interval
def insert_interval(intervals, new):
    result = []
    for i, interval in enumerate(intervals):
        if interval[1] < new[0]:
            result.append(interval)
        elif interval[0] > new[1]:
            result.append(new)
            return result + intervals[i:]
        else:
            new = [min(new[0], interval[0]), max(new[1], interval[1])]
    result.append(new)
    return result

print("Insert [2,5] into [[1,3],[6,9]]:")
print(" ", insert_interval([[1,3],[6,9]], [2,5]))  # [[1,5],[6,9]]

In [ ]:
# Non-overlapping Intervals (minimum removals)
def erase_overlap_intervals(intervals):
    """Greedy: sort by end, keep intervals that don't overlap. Time: O(n log n)"""
    intervals.sort(key=lambda x: x[1])
    count = 0
    prev_end = float('-inf')
    for start, end in intervals:
        if start >= prev_end:
            prev_end = end
        else:
            count += 1
    return count

print("Min removals for non-overlap:")
print(" ", erase_overlap_intervals([[1,2],[2,3],[3,4],[1,3]]))  # 1

In [ ]:
# Meeting Rooms II (minimum rooms needed)
def min_meeting_rooms(intervals):
    """Use a min-heap to track room end times. Time: O(n log n)"""
    if not intervals: return 0
    intervals.sort(key=lambda x: x[0])
    heap = [intervals[0][1]]  # End time of first meeting
    for start, end in intervals[1:]:
        if start >= heap[0]:
            heapq.heappop(heap)  # Room freed up
        heapq.heappush(heap, end)
    return len(heap)

print("Min meeting rooms:")
print(" ", min_meeting_rooms([[0,30],[5,10],[15,20]]))  # 2

In [ ]:
# Sort Colors (Dutch National Flag)
def sort_colors(nums):
    """Three-pointer partition. O(n) time, O(1) space, single pass."""
    lo, mid, hi = 0, 0, len(nums) - 1
    while mid <= hi:
        if nums[mid] == 0:
            nums[lo], nums[mid] = nums[mid], nums[lo]
            lo += 1; mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[hi] = nums[hi], nums[mid]
            hi -= 1
    return nums

print("Sort Colors:", sort_colors([2,0,2,1,1,0]))  # [0,0,1,1,2,2]

---
## 📦 Heap / Priority Queue (heapq)

In [ ]:
# heapq basics — Python only has MIN heap
h = []
heapq.heappush(h, 5)
heapq.heappush(h, 1)
heapq.heappush(h, 3)
print("Min:", heapq.heappop(h))  # 1

# MAX heap trick: negate values
max_heap = []
for val in [5, 1, 3]:
    heapq.heappush(max_heap, -val)
print("Max:", -heapq.heappop(max_heap))  # 5

# heapify — convert list to heap in O(n)
nums = [5, 1, 3, 7, 2]
heapq.heapify(nums)
print("Heapified:", nums)  # [1, 2, 3, 7, 5]

# nlargest / nsmallest
print("3 largest:", heapq.nlargest(3, [1,5,3,7,2]))   # [7, 5, 3]
print("3 smallest:", heapq.nsmallest(3, [1,5,3,7,2]))  # [1, 2, 3]

In [ ]:
# Kth Largest Element
def find_kth_largest(nums, k):
    """Min heap of size k. Time: O(n log k)"""
    heap = nums[:k]
    heapq.heapify(heap)
    for num in nums[k:]:
        if num > heap[0]:
            heapq.heapreplace(heap, num)
    return heap[0]

print("Kth Largest:")
print("  [3,2,1,5,6,4], k=2:", find_kth_largest([3,2,1,5,6,4], 2))  # 5

In [ ]:
# K Closest Points to Origin
def k_closest(points, k):
    """Max heap of size k (negate distances for max heap). Time: O(n log k)"""
    heap = []
    for x, y in points:
        dist = -(x*x + y*y)  # Negate for max heap
        if len(heap) < k:
            heapq.heappush(heap, (dist, x, y))
        elif dist > heap[0][0]:
            heapq.heapreplace(heap, (dist, x, y))
    return [[x, y] for _, x, y in heap]

print("K Closest Points:")
print(" ", k_closest([[1,3],[-2,2],[5,8],[0,1]], 2))  # [[0,1],[-2,2]] or similar

In [ ]:
# Merge K Sorted Lists
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def merge_k_sorted(lists):
    """Use a min-heap to always pick the smallest element. Time: O(N log k)"""
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    
    dummy = ListNode(0)
    current = dummy
    while heap:
        val, i, node = heapq.heappop(heap)
        current.next = node
        current = current.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    
    return dummy.next

print("Merge K Sorted Lists: (conceptual — uses ListNode)")

In [ ]:
# Find Median from Data Stream (Two Heaps)
class MedianFinder:
    """Two heaps: max_heap for lower half, min_heap for upper half.
    Median = top of max_heap (or average of both tops).
    addNum: O(log n), findMedian: O(1)"""
    def __init__(self):
        self.lo = []  # max heap (negated)
        self.hi = []  # min heap
    
    def addNum(self, num):
        heapq.heappush(self.lo, -num)
        heapq.heappush(self.hi, -heapq.heappop(self.lo))
        if len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))
    
    def findMedian(self):
        if len(self.lo) > len(self.hi):
            return -self.lo[0]
        return (-self.lo[0] + self.hi[0]) / 2

mf = MedianFinder()
for num in [1, 2, 3, 4, 5]:
    mf.addNum(num)
    print(f"  After adding {num}: median = {mf.findMedian()}")

---
## 🏆 Summary

| Problem | Technique | Time |
|---------|-----------|------|
| Merge Intervals | Sort + merge | O(n log n) |
| Insert Interval | Linear scan | O(n) |
| Non-overlapping | Greedy + sort by end | O(n log n) |
| Meeting Rooms II | Min heap | O(n log n) |
| Sort Colors | Three-pointer | O(n) |
| Kth Largest | Min heap size k | O(n log k) |
| K Closest Points | Max heap size k | O(n log k) |
| Merge K Sorted | Min heap | O(N log k) |
| Median Stream | Two heaps | O(log n) per add |